In [1]:
from playwright.async_api import async_playwright
from pathlib import Path
from bs4 import BeautifulSoup
from lxml import html
import re
import asyncio
from pydantic import BaseModel

In [2]:
auth_dir = Path("..", "..", "playwright", ".auth")

results_count_xpath = "/html/body/div[1]/div[1]/div/main/div/div/h1"
cards_layout_xpath = "/html/body/div[1]/div[1]/div/main/div/div/div/div[1]"

In [4]:
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=False)
    context = await browser.new_context(storage_state=auth_dir / "magnit.json")
    page = await context.new_page()
    search_url = "https://magnit.ru/search?term=масло подсолнечное"
    await page.goto(search_url, wait_until="domcontentloaded", timeout=60000)
    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
    content = await page.content()
    soup = BeautifulSoup(content, "html.parser")

    with open("magnit.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    

In [5]:
with open("magnit.html", "r", encoding="utf-8") as f:
    html_magnit = f.read()

tree = html.fromstring(bytes(html_magnit, encoding="utf-8"))

In [6]:
cards_layout = tree.xpath(cards_layout_xpath)[0]

cards = cards_layout.xpath(".//article")
print(f"Найдено карточек: {len(cards)}")

EXTRACT_RATIO = 0.35
need_to_extract = min(int(len(cards) * EXTRACT_RATIO), 20)

class Product(BaseModel):
    title: str
    price: str
    image_url: str
    product_url: str
    kkal: str = None
    protein: str = None
    fats: str = None
    carbs: str = None

products = []

for i, card in enumerate(cards[:need_to_extract]):
    title_el = card.xpath("./a/div[2]/div[1]/div[2]")
    price_el = card.xpath("./a/div[2]/div[1]/div[1]/div/span")
    image_el = card.xpath("./a/div[1]/div/img")
    href_el = card.xpath("./a/@href")
    
    title = title_el[0].text_content().strip() if title_el else "Без названия"
    price_text = price_el[0].text_content().strip() if price_el else "Цена не указана"
    image_src = image_el[0].get("src") if image_el else "Изображение не найдено"
    href = href_el[0] if href_el else "Ссылка не найдена"

    product = Product(
        title=title,
        price=price_text,
        image_url=image_src,
        product_url=f"https://magnit.ru{href}"
    )
    print(f"{i+1}: {product.title} - {product.price} - {product.product_url}")
    products.append(product)

Найдено карточек: 21
1: Масло подсолнечное Altero Golden с добавлением оливкового 810мл - 139 ₽ - https://magnit.ru/product/1799900041-altero_gold_maslo_rast_smes_pods_s_oliv_0_81l_efko_15?shopCode=784507&shopType=express
2: Масло подсолнечное Олейна 1л - 139 ₽ - https://magnit.ru/product/3020860002-maslo_podsolnechnoe_oleyna_1l?shopCode=784507&shopType=express
3: Масло подсолнечное 900мл - 109.99 ₽ - https://magnit.ru/product/1000029331-maslo_podsolnechnoe_rafinirovannoe_dezodorirovannoe_900ml?shopCode=784507&shopType=express
4: Масло подсолнечное Слобода рафинированное 1л - 156.99 ₽ - https://magnit.ru/product/2400010295-maslo_podsolnechnoe_sloboda_rafinirovannoe_1l?shopCode=784507&shopType=express
5: Масло подсолнечное Благо рафинированное дезодорированное 1л - 129 ₽ - https://magnit.ru/product/1000176577-blago_maslo_podsoln_raf_dez_1l_pl_but_blago_15?shopCode=784507&shopType=express
6: Масло подсолнечное Золотая семечка рафинированное 1л - 129.99 ₽ - https://magnit.ru/product/96396

In [9]:
PRODUCT_DETAIL_NUTRITION_LAYOUT_FULL_XPATH = "/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]"
KKAL_TEXT_FULL_XPATH = "/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[1]/div[1]"
KKAL_FULL_XPATH = "/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[1]/div[2]"
PROTEIN_TEXT_FULL_XPATH = "/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[2]/div[1]"
PROTEIN_FULL_XPATH = "/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[2]/div[2]"
FATS_TEXT_FULL_XPATH = "/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[3]/div[1]"
FATS_FULL_XPATH = "/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[3]/div[2]"
CARBS_TEXT_FULL_XPATH = "/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[4]/div[1]"
CARBS_FULL_XPATH = "/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[4]/div[2]"


# <section class="product-details-nutrition-facts" data-test-id="product-details-nutrition" data-v-10973777="" data-v-0ab9bba4=""><div data-test-id="v-product-details-nutrition-name" class="pl-text product-details-nutrition-facts__title" style="--_f:var(--pl-font-caption-large-accent);--_f-l:var(--pl-font-body-large-accent);--_c:var(--pl-text-secondary);" data-v-230884a4="" data-v-10973777=""><!--[-->Пищевая ценность в 100 граммах<!--]--></div><div class="product-details-nutrition-facts__list" data-v-10973777=""><!--[--><div class="product-details-nutrition-facts__list-item" data-v-10973777=""><div data-test-id="v-product-details-nutrition-fact-name" class="pl-text product-details-nutrition-facts__list-item__title" style="--_f:var(--pl-font-caption-large-regular);--_c:var(--pl-text-secondary);" data-v-230884a4="" data-v-10973777=""><!--[-->Ккал<!--]--></div><div data-test-id="v-product-details-nutrition-fact-value" class="pl-text" style="--_f:var(--pl-font-body-large-accent);--_c:var(--pl-text-primary);" data-v-230884a4="" data-v-10973777=""><!--[-->899<!--]--></div></div><div class="product-details-nutrition-facts__list-item" data-v-10973777=""><div data-test-id="v-product-details-nutrition-fact-name" class="pl-text product-details-nutrition-facts__list-item__title" style="--_f:var(--pl-font-caption-large-regular);--_c:var(--pl-text-secondary);" data-v-230884a4="" data-v-10973777=""><!--[-->Жиры<!--]--></div><div data-test-id="v-product-details-nutrition-fact-value" class="pl-text" style="--_f:var(--pl-font-body-large-accent);--_c:var(--pl-text-primary);" data-v-230884a4="" data-v-10973777=""><!--[-->99.9<!--]--></div></div><!--]--></div></section>

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=False)
    context = await browser.new_context(storage_state=auth_dir / "magnit.json")
    page = await context.new_page()
    for product in products:
        await page.goto(product.product_url, wait_until="domcontentloaded", timeout=60000)
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        content = await page.content()
        soup = BeautifulSoup(content, "html.parser")

        product_tree = html.fromstring(bytes(str(soup), encoding="utf-8"))

        product_detail_nutrition_layout = product_tree.xpath(PRODUCT_DETAIL_NUTRITION_LAYOUT_FULL_XPATH)
        
        kkal = protein = fats = carbs = None
        if product_detail_nutrition_layout:
            section = product_detail_nutrition_layout[0]

            rows = section.xpath(".//div/div")
            for row in rows:
                name_el = row.xpath(".//div[1]")
                value_el = row.xpath(".//div[2]")
                if not name_el or not value_el:
                    continue
                name = name_el[0].text_content().strip().lower()
                value = value_el[0].text_content().strip().lower()
                if "ккал" in name:
                    kkal = value
                elif "жир" in name:
                    fats = value
                elif "углевод" in name:
                    carbs = value
                elif "белок" in name:
                    protein = value

        product.kkal = kkal
        product.protein = protein
        product.fats = fats
        product.carbs = carbs

In [10]:
for product in products:
    print(f"{product.title} - {product.price} - {product.kkal} - {product.protein} - {product.fats} - {product.carbs} - {product.product_url}")

Масло подсолнечное Altero Golden с добавлением оливкового 810мл - 139 ₽ - 899 - None - 99.9 - None - https://magnit.ru/product/1799900041-altero_gold_maslo_rast_smes_pods_s_oliv_0_81l_efko_15?shopCode=784507&shopType=express
Масло подсолнечное Олейна 1л - 139 ₽ - 899 - None - 99.9 - None - https://magnit.ru/product/3020860002-maslo_podsolnechnoe_oleyna_1l?shopCode=784507&shopType=express
Масло подсолнечное 900мл - 109.99 ₽ - None - None - None - None - https://magnit.ru/product/1000029331-maslo_podsolnechnoe_rafinirovannoe_dezodorirovannoe_900ml?shopCode=784507&shopType=express
Масло подсолнечное Слобода рафинированное 1л - 156.99 ₽ - 899 - None - 99.9 - None - https://magnit.ru/product/2400010295-maslo_podsolnechnoe_sloboda_rafinirovannoe_1l?shopCode=784507&shopType=express
Масло подсолнечное Благо рафинированное дезодорированное 1л - 129 ₽ - 899 - None - 99.9 - None - https://magnit.ru/product/1000176577-blago_maslo_podsoln_raf_dez_1l_pl_but_blago_15?shopCode=784507&shopType=express
